# Lab: Inference Metrics, Goodput & Monitoring

This lab builds intuition for the metrics that matter in LLM serving:
- **TTFT** (Time To First Token) and **ITL** (Inter-Token Latency) distributions
- Why percentiles reveal what averages hide
- Goodput vs raw throughput under SLO constraints
- GPU-level signals (KV cache pressure, preemption)
- Cost modeling at different utilization levels
- Open-loop vs closed-loop benchmarking pitfalls

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from dataclasses import dataclass, field
from typing import List

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
print('Setup complete.')

## Simulating LLM Inference Requests

Real inference traffic exhibits heavy-tailed latency distributions:
- **TTFT** follows a lognormal (prefill compute varies with prompt length)
- **ITL** follows a gamma (decode steps are more regular but still variable)
- **Output length** follows a geometric (most responses are short, some are very long)

In [ ]:
@dataclass
class RequestTrace:
    """Single inference request with timing breakdown."""
    request_id: int
    prompt_tokens: int
    output_tokens: int
    ttft_ms: float          # Time to first token
    itl_ms: np.ndarray      # Per-token inter-token latencies
    arrival_time: float     # Seconds from start

    @property
    def total_latency_ms(self) -> float:
        return self.ttft_ms + self.itl_ms.sum()

    @property
    def mean_itl_ms(self) -> float:
        return self.itl_ms.mean()


def generate_traces(n: int = 1000) -> List[RequestTrace]:
    traces = []
    for i in range(n):
        prompt_tokens = int(np.random.lognormal(mean=5.5, sigma=0.8))  # ~250 avg
        output_tokens = max(1, int(np.random.geometric(p=0.02)))       # ~50 avg
        # TTFT scales with prompt length (prefill bound)
        ttft = np.random.lognormal(mean=np.log(80 + prompt_tokens * 0.3), sigma=0.4)
        # ITL: gamma-distributed decode steps
        itl = np.random.gamma(shape=3.0, scale=15.0, size=output_tokens)
        arrival = i * 0.1 + np.random.exponential(0.02)  # ~10 req/s
        traces.append(RequestTrace(i, prompt_tokens, output_tokens, ttft, itl, arrival))
    return traces


traces = generate_traces(1000)
print(f'Generated {len(traces)} request traces')
print(f'Avg prompt tokens: {np.mean([t.prompt_tokens for t in traces]):.0f}')
print(f'Avg output tokens: {np.mean([t.output_tokens for t in traces]):.0f}')
print(f'Avg TTFT: {np.mean([t.ttft_ms for t in traces]):.1f} ms')
print(f'Avg ITL: {np.mean([t.mean_itl_ms for t in traces]):.1f} ms')

## Percentile Analysis: Why Averages Lie

The mean TTFT might be 200ms, but if p99 is 2000ms, 1% of your users wait 10x longer.
SLOs must be defined at tail percentiles (p95/p99), not means.

In [ ]:
ttft_values = np.array([t.ttft_ms for t in traces])
itl_values = np.concatenate([t.itl_ms for t in traces])

percentiles = [50, 75, 90, 95, 99]
ttft_pcts = np.percentile(ttft_values, percentiles)
itl_pcts = np.percentile(itl_values, percentiles)

print('TTFT Percentiles (ms):')
for p, v in zip(percentiles, ttft_pcts):
    print(f'  p{p:02d}: {v:>8.1f} ms')
print(f'  mean: {ttft_values.mean():>7.1f} ms')
print(f'\nITL Percentiles (ms):')
for p, v in zip(percentiles, itl_pcts):
    print(f'  p{p:02d}: {v:>8.1f} ms')
print(f'  mean: {itl_values.mean():>7.1f} ms')

# Histogram with percentile lines
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2563eb', '#7c3aed', '#dc2626', '#ea580c', '#0d9488']

axes[0].hist(ttft_values, bins=60, alpha=0.7, color='#dbeafe', edgecolor='#1e293b')
for p, v, c in zip(percentiles, ttft_pcts, colors):
    axes[0].axvline(v, color=c, linestyle='--', linewidth=1.5, label=f'p{p}={v:.0f}ms')
axes[0].set_xlabel('TTFT (ms)')
axes[0].set_ylabel('Count')
axes[0].set_title('TTFT Distribution with Percentile Lines')
axes[0].legend(fontsize=9)

axes[1].hist(itl_values, bins=60, alpha=0.7, color='#dcfce7', edgecolor='#1e293b')
for p, v, c in zip(percentiles, itl_pcts, colors):
    axes[1].axvline(v, color=c, linestyle='--', linewidth=1.5, label=f'p{p}={v:.0f}ms')
axes[1].set_xlabel('ITL (ms)')
axes[1].set_ylabel('Count')
axes[1].set_title('ITL Distribution with Percentile Lines')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig('percentile_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## Computing Goodput

**Goodput** = tokens generated that meet ALL SLO constraints / total tokens generated.

A system can have high raw throughput (tokens/sec) but low goodput if many tokens
are produced outside SLO bounds. This is the metric that actually correlates with user satisfaction.

In [ ]:
# Define SLOs
SLO_TTFT_MS = 500.0   # First token must arrive within 500ms
SLO_ITL_MS = 100.0    # Each subsequent token within 100ms

def compute_goodput(traces: List[RequestTrace], slo_ttft: float, slo_itl: float):
    """Compute goodput: fraction of tokens meeting ALL SLOs."""
    total_tokens = 0
    good_tokens = 0
    for t in traces:
        total_tokens += t.output_tokens
        if t.ttft_ms <= slo_ttft:
            # Count tokens where ITL is within SLO
            good_tokens += int((t.itl_ms <= slo_itl).sum())
        # If TTFT violated, ALL tokens from this request are bad
    return good_tokens / total_tokens if total_tokens > 0 else 0.0


goodput = compute_goodput(traces, SLO_TTFT_MS, SLO_ITL_MS)
total_tokens = sum(t.output_tokens for t in traces)
total_time_s = max(t.arrival_time + t.total_latency_ms/1000 for t in traces)
raw_throughput = total_tokens / total_time_s

print(f'SLOs: TTFT < {SLO_TTFT_MS}ms, ITL < {SLO_ITL_MS}ms')
print(f'Total tokens generated: {total_tokens:,}')
print(f'Raw throughput: {raw_throughput:.1f} tokens/sec')
print(f'Goodput: {goodput*100:.1f}% of tokens meet SLOs')
print(f'Effective throughput: {raw_throughput * goodput:.1f} good tokens/sec')

In [ ]:
# Simulate throughput vs goodput divergence under increasing load
load_multipliers = np.linspace(0.5, 3.0, 20)
raw_tps = []
good_tps = []

for mult in load_multipliers:
    # Under higher load, latencies increase (queuing)
    loaded_traces = []
    for t in traces[:200]:  # Subset for speed
        load_factor = 1.0 + (mult - 1.0) * 0.8  # Latency grows sub-linearly
        new_ttft = t.ttft_ms * load_factor + np.random.exponential(20 * mult)
        new_itl = t.itl_ms * load_factor
        loaded_traces.append(RequestTrace(
            t.request_id, t.prompt_tokens, t.output_tokens,
            new_ttft, new_itl, t.arrival_time / mult
        ))
    tok = sum(t.output_tokens for t in loaded_traces)
    dur = max(t.arrival_time + t.total_latency_ms/1000 for t in loaded_traces)
    raw = tok / dur
    gp = compute_goodput(loaded_traces, SLO_TTFT_MS, SLO_ITL_MS)
    raw_tps.append(raw)
    good_tps.append(raw * gp)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(load_multipliers, raw_tps, 'b-o', markersize=5, label='Raw Throughput (tok/s)')
ax.plot(load_multipliers, good_tps, 'r-s', markersize=5, label='Goodput (good tok/s)')
ax.fill_between(load_multipliers, good_tps, raw_tps, alpha=0.15, color='red', label='SLO-violating tokens')
ax.axvline(1.0, color='gray', linestyle=':', label='Baseline load')
ax.set_xlabel('Load Multiplier (1.0 = baseline)')
ax.set_ylabel('Tokens / Second')
ax.set_title('Raw Throughput vs Goodput Under Increasing Load')
ax.legend()
plt.tight_layout()
plt.savefig('throughput_vs_goodput.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nKey insight: raw throughput keeps climbing, but goodput plateaus and drops.')

## GPU Metrics Simulation

KV cache occupancy is the critical GPU-side metric for LLM serving.
When it exceeds ~85%, the scheduler must preempt (evict) sequences to avoid OOM,
destroying work already done and spiking tail latency.

In [ ]:
# Simulate KV cache occupancy over 60 seconds
timesteps = np.arange(0, 60, 0.1)  # 100ms resolution
kv_occupancy = np.zeros_like(timesteps)
preemption_events = []
ALERT_THRESHOLD = 0.85

# Start at 40% occupancy, grows with requests
base = 0.40
for i, t in enumerate(timesteps):
    # Occupancy grows as new requests arrive and generate tokens
    growth = 0.012 * t + 0.05 * np.sin(t * 0.5)  # Trend + bursts
    noise = np.random.normal(0, 0.02)
    kv_occupancy[i] = base + growth + noise
    
    # Preemption: if over threshold, evict and drop
    if kv_occupancy[i] > ALERT_THRESHOLD:
        preemption_events.append((t, kv_occupancy[i]))
        kv_occupancy[i] -= 0.15  # Evict ~15% of cache
    kv_occupancy[i] = np.clip(kv_occupancy[i], 0, 1.0)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(timesteps, kv_occupancy * 100, color='#2563eb', linewidth=1.5)
ax.axhline(ALERT_THRESHOLD * 100, color='#dc2626', linestyle='--', linewidth=2, label=f'Alert threshold ({ALERT_THRESHOLD*100:.0f}%)')
if preemption_events:
    pe_times = [e[0] for e in preemption_events]
    pe_vals = [e[1]*100 for e in preemption_events]
    ax.scatter(pe_times, pe_vals, color='red', zorder=5, s=40, label=f'Preemption events ({len(preemption_events)})')
ax.set_xlabel('Time (seconds)')
ax.set_ylabel('KV Cache Occupancy (%)')
ax.set_title('KV Cache Pressure Over Time (with Preemption Events)')
ax.set_ylim(0, 100)
ax.legend()
plt.tight_layout()
plt.savefig('kv_cache_pressure.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Total preemption events in 60s: {len(preemption_events)}')
print(f'Each preemption destroys partial generation work and spikes TTFT for re-queued requests.')

## Cost Per Million Tokens

Cost efficiency depends on GPU utilization. At low utilization you pay for idle silicon.
At very high utilization, SLO violations and preemptions make effective cost *worse*
because you pay for tokens that don't count toward goodput.

In [ ]:
@dataclass
class CostModel:
    """Model $/M tokens at different utilization levels."""
    gpu_cost_per_hour: float = 3.50   # e.g., A100 80GB on-demand
    max_throughput_tps: float = 2000.0  # Peak tokens/sec at 100% util

    def cost_per_million(self, utilization: float) -> float:
        """Raw cost per million tokens at given utilization."""
        effective_tps = self.max_throughput_tps * utilization
        tokens_per_hour = effective_tps * 3600
        if tokens_per_hour == 0:
            return float('inf')
        return (self.gpu_cost_per_hour / tokens_per_hour) * 1_000_000

    def effective_cost_per_million(self, utilization: float, goodput_ratio: float) -> float:
        """Cost per million GOOD tokens (accounting for SLO violations)."""
        raw = self.cost_per_million(utilization)
        return raw / goodput_ratio if goodput_ratio > 0 else float('inf')


model = CostModel()
utils = np.linspace(0.1, 0.99, 50)
# Goodput ratio drops at high utilization due to queuing
goodput_ratios = np.clip(1.0 - 0.5 * (utils - 0.7)**2 * (utils > 0.7), 0.5, 1.0)

raw_costs = [model.cost_per_million(u) for u in utils]
effective_costs = [model.effective_cost_per_million(u, g) for u, g in zip(utils, goodput_ratios)]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(utils * 100, raw_costs, 'b-', linewidth=2, label='Raw $/M tokens')
ax.plot(utils * 100, effective_costs, 'r--', linewidth=2, label='Effective $/M good tokens')
# Find optimal point
opt_idx = np.argmin(effective_costs)
ax.scatter([utils[opt_idx]*100], [effective_costs[opt_idx]], color='green', s=100, zorder=5,
           label=f'Optimal: {utils[opt_idx]*100:.0f}% util = ${effective_costs[opt_idx]:.2f}/M')
ax.set_xlabel('GPU Utilization (%)')
ax.set_ylabel('Cost ($ / Million Tokens)')
ax.set_title('Cost Per Million Tokens vs GPU Utilization')
ax.legend()
ax.set_ylim(0, max(raw_costs[:10]) * 1.1)
plt.tight_layout()
plt.savefig('cost_per_million.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nOptimal utilization: {utils[opt_idx]*100:.0f}%')
print(f'Raw cost at optimal: ${raw_costs[opt_idx]:.3f}/M tokens')
print(f'Effective cost at optimal: ${effective_costs[opt_idx]:.3f}/M good tokens')

## Benchmarking: Open vs Closed Loop

**Closed-loop**: send next request only after previous completes. This *coordinating* pattern
artificially reduces concurrency when the system is slow, hiding true tail latency.

**Open-loop**: send requests at a fixed arrival rate regardless of completion. This reveals
how the system behaves under sustained load (queuing builds up naturally).

Most real traffic is open-loop. Always benchmark with open-loop to see true tail behavior.

In [ ]:
def simulate_closed_loop(n_requests: int, service_time_fn, think_time: float = 0.01):
    """Closed loop: wait for response + think_time before sending next."""
    latencies = []
    clock = 0.0
    for _ in range(n_requests):
        service = service_time_fn()
        latencies.append(service)
        clock += service + think_time
    return np.array(latencies)


def simulate_open_loop(n_requests: int, service_time_fn, arrival_rate: float):
    """Open loop: fixed arrival rate, queuing builds naturally."""
    arrivals = np.cumsum(np.random.exponential(1.0/arrival_rate, n_requests))
    latencies = []
    finish_time = 0.0
    for arr in arrivals:
        wait = max(0, finish_time - arr)  # Queue wait
        service = service_time_fn()
        total = wait + service
        latencies.append(total)
        finish_time = arr + total
    return np.array(latencies)


# Service time: lognormal with occasional spikes
def service_time():
    base = np.random.lognormal(mean=np.log(0.05), sigma=0.5)  # ~50ms
    if np.random.random() < 0.05:  # 5% spike
        base *= 5
    return base

N = 500
closed = simulate_closed_loop(N, service_time) * 1000  # to ms
opened = simulate_open_loop(N, service_time, arrival_rate=15) * 1000

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, data, title, color in [
    (axes[0], closed, 'Closed-Loop', '#dbeafe'),
    (axes[1], opened, 'Open-Loop', '#ffe4e6')
]:
    ax.hist(data, bins=50, alpha=0.8, color=color, edgecolor='#1e293b')
    ax.axvline(np.percentile(data, 99), color='red', linestyle='--', linewidth=2,
               label=f'p99={np.percentile(data, 99):.0f}ms')
    ax.axvline(np.percentile(data, 50), color='blue', linestyle='--', linewidth=2,
               label=f'p50={np.percentile(data, 50):.0f}ms')
    ax.set_title(f'{title} Latency Distribution')
    ax.set_xlabel('Latency (ms)')
    ax.set_ylabel('Count')
    ax.legend()

plt.tight_layout()
plt.savefig('open_vs_closed_loop.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Closed-loop: p50={np.percentile(closed,50):.0f}ms, p99={np.percentile(closed,99):.0f}ms')
print(f'Open-loop:   p50={np.percentile(opened,50):.0f}ms, p99={np.percentile(opened,99):.0f}ms')
print(f'\nTail latency ratio (p99 open/closed): {np.percentile(opened,99)/np.percentile(closed,99):.1f}x')
print('Open-loop reveals the true queuing behavior that closed-loop hides.')

## Key Takeaways

1. **Percentiles over averages**: p95/p99 reveal the user experience that means hide.
2. **Goodput > throughput**: Raw tokens/sec is meaningless if tokens violate SLOs.
3. **KV cache is the bottleneck**: Monitor occupancy; preemption events signal capacity limits.
4. **Cost has a U-curve**: Too-low utilization wastes money; too-high causes SLO violations that raise effective cost.
5. **Open-loop benchmarking is mandatory**: Closed-loop under-reports tail latency by 3-10x.
6. **Compound metrics matter**: The best single number is `effective_cost_per_good_token` which combines utilization, throughput, and SLO compliance.